# Bank Deposit Classification: Model Comparison

This notebook compares several supervised-learning algorithms for predicting whether a bank client subscribes to a term deposit.

The workflow uses the local dataset:

```text
Machine-Learning-Course/datasets/bank.csv
```

## Learning Objectives

After completing this notebook, students should be able to:

1. inspect a mixed numerical and categorical dataset;
2. separate features from a binary target;
3. create a stratified training and test split;
4. preprocess numerical and categorical columns without data leakage;
5. compare multiple classifiers under the same evaluation protocol;
6. interpret accuracy, precision, recall, F1 score, ROC AUC, and log loss;
7. inspect a confusion matrix and detailed classification report;
8. identify limitations that affect model selection and deployment.

## Prediction Task

The target column is `deposit`:

- `yes` indicates that the client subscribed;
- `no` indicates that the client did not subscribe.

The notebook maps these labels to:

- `1` for `yes`;
- `0` for `no`.

## 1. Import the required libraries

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## 2. Load the local dataset

The notebook is stored in `Machine-Learning-Course/notebooks`, so the dataset is referenced through a portable relative path.

In [ ]:
dataset_path = Path("../datasets/bank.csv")

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path.resolve()}"
    )

data = pd.read_csv(dataset_path)

print("Dataset shape:", data.shape)
display(data.head())

## 3. Inspect data quality and class balance

The checks below confirm the column types, missing-value counts, duplicate rows, and target distribution before modeling.

In [ ]:
print("Column data types:")
display(data.dtypes.to_frame("dtype"))

missing_values = data.isna().sum()
missing_values = missing_values[missing_values > 0]

print("Missing values:")
if missing_values.empty:
    print("No missing values detected.")
else:
    display(missing_values.to_frame("missing_count"))

print("Duplicate rows:", int(data.duplicated().sum()))

target_counts = (
    data["deposit"]
    .value_counts()
    .rename_axis("deposit")
    .to_frame("count")
)

target_counts["percentage"] = (
    target_counts["count"]
    / target_counts["count"].sum()
    * 100
)

display(target_counts)

## 4. Visualize the target distribution

In [ ]:
target_counts["count"].plot(
    kind="bar",
    rot=0,
)

plt.title("Bank Deposit Target Distribution")
plt.xlabel("Deposit Outcome")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

## 5. Separate features and target

The target is converted to integer labels. All remaining columns are retained as input features.

In [ ]:
target_mapping = {
    "no": 0,
    "yes": 1,
}

X = data.drop(columns="deposit")
y = data["deposit"].map(target_mapping)

if y.isna().any():
    unexpected_labels = sorted(
        data.loc[y.isna(), "deposit"]
        .astype(str)
        .unique()
        .tolist()
    )
    raise ValueError(
        f"Unexpected target labels: {unexpected_labels}"
    )

y = y.astype(int)

categorical_columns = (
    X.select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

numerical_columns = (
    X.select_dtypes(include=np.number)
    .columns
    .tolist()
)

print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)
print("Feature count:", X.shape[1])

## 6. Create a stratified train-test split

Stratification preserves approximately the same proportion of `yes` and `no` outcomes in both subsets.

Preprocessing is fitted only on the training data through scikit-learn pipelines. This prevents information from the test set from influencing the learned encodings or scaling parameters.

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y,
    )
)

print("Training records:", len(X_train))
print("Test records:", len(X_test))

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True).sort_index())

## 7. Define the preprocessing pipeline

Numerical features are:

1. imputed with their median if missing values appear;
2. standardized to zero mean and unit variance.

Categorical features are:

1. imputed with their most frequent category if needed;
2. one-hot encoded.

`handle_unknown="ignore"` allows the trained pipeline to process categories that appear only in future data.

In [ ]:
def make_preprocessor():
    numerical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numerical",
                numerical_pipeline,
                numerical_columns,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns,
            ),
        ],
        remainder="drop",
    )

## 8. Define the classifiers

The comparison includes linear, distance-based, probabilistic, tree-based, ensemble, and kernel-based methods.

Each classifier receives an independent copy of the same preprocessing design.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2_000,
        random_state=42,
    ),
    "K-Nearest Neighbors": KNeighborsClassifier(
        n_neighbors=5,
    ),
    "Support Vector Machine": SVC(
        kernel="rbf",
        C=1.0,
        probability=True,
        random_state=42,
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
    "Gaussian Naive Bayes": GaussianNB(),
    "Linear Discriminant Analysis": (
        LinearDiscriminantAnalysis()
    ),
}

pipelines = {
    name: Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor(),
            ),
            (
                "classifier",
                clone(model),
            ),
        ]
    )
    for name, model in models.items()
}

print("Models to evaluate:", len(pipelines))
for model_name in pipelines:
    print("-", model_name)

## 9. Train and evaluate all models

The models are compared using:

- **Accuracy:** proportion of all predictions that are correct.
- **Precision:** proportion of predicted subscribers that are actual subscribers.
- **Recall:** proportion of actual subscribers detected by the model.
- **F1 score:** harmonic mean of precision and recall.
- **ROC AUC:** ability to rank positive examples above negative examples across thresholds.
- **Log loss:** quality and confidence of predicted probabilities; lower values are better.
- **Fit time:** approximate training time on the current computer.

In [ ]:
results = []
fitted_pipelines = {}

for model_name, pipeline in pipelines.items():
    start_time = perf_counter()
    pipeline.fit(X_train, y_train)
    fit_seconds = perf_counter() - start_time

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(
        X_test
    )[:, 1]

    results.append(
        {
            "Classifier": model_name,
            "Accuracy": accuracy_score(
                y_test,
                predictions,
            ),
            "Precision": precision_score(
                y_test,
                predictions,
                zero_division=0,
            ),
            "Recall": recall_score(
                y_test,
                predictions,
                zero_division=0,
            ),
            "F1": f1_score(
                y_test,
                predictions,
                zero_division=0,
            ),
            "ROC AUC": roc_auc_score(
                y_test,
                probabilities,
            ),
            "Log Loss": log_loss(
                y_test,
                probabilities,
            ),
            "Fit Time (s)": fit_seconds,
        }
    )

    fitted_pipelines[model_name] = pipeline

results_table = (
    pd.DataFrame(results)
    .sort_values(
        by=["ROC AUC", "Log Loss"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(
    results_table.style.format(
        {
            "Accuracy": "{:.4f}",
            "Precision": "{:.4f}",
            "Recall": "{:.4f}",
            "F1": "{:.4f}",
            "ROC AUC": "{:.4f}",
            "Log Loss": "{:.4f}",
            "Fit Time (s)": "{:.3f}",
        }
    )
)

## 10. Compare classification performance

Higher values are better for accuracy, precision, recall, F1, and ROC AUC.

In [ ]:
performance_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC AUC",
]

(
    results_table
    .set_index("Classifier")[performance_columns]
    .plot(
        kind="bar",
        figsize=(11, 6),
    )
)

plt.title("Classifier Performance Comparison")
plt.xlabel("Classifier")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 11. Compare probability quality

Lower log loss indicates better calibrated and less overconfident probability estimates.

In [ ]:
(
    results_table
    .set_index("Classifier")["Log Loss"]
    .sort_values()
    .plot(
        kind="bar",
        figsize=(9, 5),
    )
)

plt.title("Classifier Log Loss")
plt.xlabel("Classifier")
plt.ylabel("Log Loss")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 12. Inspect the highest-ranked model

For this notebook, models are ranked primarily by ROC AUC and secondarily by log loss. This ranking rule is an instructional choice; a real project should select metrics according to business costs and operational requirements.

In [ ]:
best_model_name = results_table.loc[
    0,
    "Classifier",
]

best_pipeline = fitted_pipelines[best_model_name]
best_predictions = best_pipeline.predict(X_test)

print("Highest-ranked model:", best_model_name)
print()

print(
    classification_report(
        y_test,
        best_predictions,
        target_names=[
            "No Deposit",
            "Deposit",
        ],
        zero_division=0,
    )
)

## 13. Display the confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_predictions,
    display_labels=[
        "No Deposit",
        "Deposit",
    ],
)

plt.title(
    f"Confusion Matrix: {best_model_name}"
)
plt.tight_layout()
plt.show()

## Result Interpretation

The comparison table should not be interpreted using accuracy alone.

A model may achieve strong accuracy while missing many actual subscribers, which would reduce recall. Another model may detect more subscribers but produce more false positives, which would reduce precision.

The preferred model depends on the practical cost of:

- contacting a client who is unlikely to subscribe;
- failing to identify a client who would subscribe;
- producing poorly calibrated probabilities;
- requiring excessive training or inference time.

## Methodological Improvements over Direct Label Encoding

This notebook uses a pipeline rather than encoding all columns before the train-test split.

That design provides several advantages:

- categorical encodings are learned from training data only;
- unseen test categories are handled safely;
- numerical scaling is applied consistently;
- each classifier is evaluated under the same preprocessing protocol;
- preprocessing and prediction remain packaged together.

## Limitations

This is a teaching comparison, not a final deployment study.

A production workflow should additionally consider:

- cross-validation rather than one train-test split;
- hyperparameter optimization;
- probability calibration;
- threshold optimization;
- cost-sensitive evaluation;
- feature availability at prediction time;
- fairness and subgroup analysis;
- model interpretability;
- drift monitoring;
- independent validation data.

The test set should remain untouched during model development. Repeatedly choosing models based on the same test results can lead to optimistic conclusions.

## Exercises

1. Add stratified cross-validation.
2. Tune the random-forest and SVM hyperparameters.
3. Compare results with and without numerical scaling.
4. Plot ROC and precision-recall curves.
5. Optimize the decision threshold for recall or precision.
6. Evaluate balanced accuracy.
7. Inspect feature importance for the random forest.
8. Use permutation importance for model-independent interpretation.
9. Remove potentially unavailable features and compare performance.
10. Save the final pipeline with `joblib`.